# Session 4.1 — Deployment Readiness

## What We're Doing Today

Your app works on YOUR laptop. Today we ship it to a public URL that anyone can hit.

By the end of this session you will have:
- **Five patches applied** to `app/main.py`, `pipeline/retrieval/hyde.py`, and `pipeline/retrieval/enriched.py`
- **A local Docker smoke test** passing — the same image Community Cloud will run
- **A live public URL** at `https://<your-handle>-northbrook.streamlit.app` (or similar)
- **Phoenix traces** landing in the shared workspace under your project name

## Learning Objectives

1. **Explain** the three problems of moving an app from a laptop to a public deployment — environment drift, module-level state, credential exposure
2. **Apply** anchor-based copy-paste patches to harden an app for containerized deployment
3. **Run** a local Docker smoke test using the provided launcher scripts
4. **Deploy** a Streamlit application to Community Cloud (Secrets + per-visitor API keys)
5. **Verify** the deployed URL: chat works, sources render, Phoenix traces appear

## Session Theme

> "Ship it. The deploy is part of the work."

In [1]:
# Path setup — run this cell FIRST
# Notebooks run from their subdirectory but pipeline modules are at the project root
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

# Load environment variables from project root
from dotenv import load_dotenv
load_dotenv(_PROJECT_ROOT / ".env")

print(f"Project root: {_PROJECT_ROOT}")
print(f"Path configured: pipeline imports will work")

Project root: c:\AI-2\ai3-fullstack
Path configured: pipeline imports will work


In [2]:
# Import verification — pipeline modules used in deploy patches
# We are NOT constructing clients here. Just confirming the modules import.
from pipeline.retrieval.hyde import hyde_retrieve, generate_hypothetical_answer
from pipeline.retrieval.enriched import enriched_retrieve, generate_questions_for_chunk
from app.feedback import submit_feedback, get_feedback_summary

print("All target modules import successfully.")
print("  - pipeline.retrieval.hyde     (Patch 3 target)")
print("  - pipeline.retrieval.enriched (Patch 4 target)")
print("  - app.feedback                (Patch 5 — confirmation only)")

c:\AI-2\ai3-fullstack\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All target modules import successfully.
  - pipeline.retrieval.hyde     (Patch 3 target)
  - pipeline.retrieval.enriched (Patch 4 target)
  - app.feedback                (Patch 5 — confirmation only)


In [3]:
# Load mermaid diagram support — run once per notebook
%load_ext mermaid_magic

## Pre-Class Checklist

Before we apply any patches, confirm you have these ready (these were in the pre-class email):

- [ ] **GitHub fork** of `ai3-fullstack` exists under YOUR account
- [ ] **Your working branch** is pushed to your fork
- [ ] **Docker Desktop** is installed and running (whale icon in menu bar / system tray)
- [ ] **`.env` file** exists in the project root with `ANTHROPIC_API_KEY`, `VOYAGE_API_KEY`, and the Phoenix triple
- [ ] **App runs locally** — `streamlit run app/main.py` answers a question

If any of these are not true, fix it now. Today does not work without them.

---

## Section 1 — The Three Deployment Problems

Moving an app from your laptop to a public URL surfaces three distinct problems. Each one has a specific pattern that solves it.

| # | Problem | Pattern | Where It Lives |
|---|---------|---------|----------------|
| 1 | "Works on my machine" — env drift, deps, OS | Docker container | Already in your repo (`Dockerfile`) |
| 2 | Module-level state breaks on cold boot | Lazy client pattern | Patches 3 + 4 (`hyde.py`, `enriched.py`) |
| 3 | Public URL exposes your API credits | Sidebar key entry — visitors bring their own | Patch 2 (`app/main.py`) |

We hit all three today. The notebook walks each patch top to bottom.

### Why this matters

On your laptop, `.env` is on disk before any module loads, your Python version is fixed, and there is exactly one user (you). On Community Cloud:

- There is **no `.env` file** — configuration comes from a Secrets dashboard, but the user-supplied keys come in AFTER the app has already imported every module.
- The container is **ephemeral** — it can restart at any time. Anything that lived only in memory is gone.
- The URL is **public** — anyone can hit it, and every API call uses whatever keys the runtime has access to.

### Architecture: Laptop vs Community Cloud

In [4]:
%%mermaid
graph LR
    subgraph LAPTOP["Your Laptop"]
        A["streamlit run"] --> B["ChromaDB on disk"]
        A --> C[".env on disk"]
        C --> D["All keys preloaded"]
        E["One user (you)"]
    end
    subgraph CLOUD["Community Cloud"]
        F["share.streamlit.io"] --> G["ChromaDB rebuilt on cold boot"]
        F --> H["Secrets dashboard - Phoenix only"]
        F --> I["Sidebar key input"]
        I --> J["Anthropic + Voyage per visitor"]
        K["Public URL - N visitors"]
    end

    style LAPTOP fill:#d4edda,stroke:#155724
    style CLOUD fill:#cce5ff,stroke:#004085

---

## Section 2 — Application File Updates

We are about to apply **5 patches** across **3 files**. Here is the map:

| Patch | File | What it does |
|-------|------|--------------|
| 1 | `app/main.py` | `pysqlite3` shim — fixes ChromaDB on Community Cloud's old sqlite3 |
| 2 | `app/main.py` | Sidebar API key entry — anti-credit-burn |
| 3 | `pipeline/retrieval/hyde.py` | Lazy `_get_client()` — defer Anthropic client construction |
| 4 | `pipeline/retrieval/enriched.py` | Same lazy pattern |
| 5 | `app/feedback.py` | NO CHANGE — already correct from 3.2; we just inspect it |

Each patch has three notebook cells:
1. **WHY** — markdown explaining the failure mode it solves
2. **COPY-PASTE** — markdown with the anchor and the code to paste
3. **VERIFY** — a short code cell that greps for the new content

All anchors are unique strings (not line numbers) — Lab 2 collision-proof.

> **Reminder:** All edits in this section happen in the **app code**, not in this notebook. Open `app/main.py`, `pipeline/retrieval/hyde.py`, and `pipeline/retrieval/enriched.py` in your editor. Use the verify cells to confirm each paste landed.

---

## Patch 1 — `app/main.py` pysqlite3 shim

### WHY

Streamlit Community Cloud's base image ships with a system `sqlite3` that is older than what ChromaDB requires. Locally on your Mac/Linux laptop, this is fine — your `sqlite3` is new enough. On Community Cloud, ChromaDB will refuse to import.

The fix: a small block of code at the very top of `app/main.py` that swaps in a newer `sqlite3` from the `pysqlite3-binary` package. The block does nothing on your laptop (the package is not installed locally), but on Community Cloud it ensures ChromaDB sees a modern `sqlite3`.

**This patch is purely defensive.** Without it, deploy fails with a cryptic ChromaDB import error. With it, deploy works.

**Where:** Immediately AFTER the existing line `sys.path.insert(0, str(_PROJECT_ROOT))` in `app/main.py`.

### COPY-PASTE

**Find this anchor** in `app/main.py`:

```
sys.path.insert(0, str(_PROJECT_ROOT))
```

**Insert the code from the next cell immediately AFTER that line (do not delete the anchor line):**


In [ ]:
# === COPY THIS BLOCK INTO app/main.py — DO NOT RUN THIS CELL ===
# Goes immediately AFTER `sys.path.insert(0, str(_PROJECT_ROOT))`

# pysqlite3 shim — required for ChromaDB on Streamlit Community Cloud.
# Community Cloud's system sqlite3 is older than ChromaDB requires.
# pysqlite3-binary ships a newer sqlite3; we swap it in before chromadb imports.
# Local Mac dev skips this gracefully (pysqlite3 not installed).
try:
    __import__('pysqlite3')
    import sys
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
except ImportError:
    pass


**For Claude Code users, paste this prompt:**

> In `app/main.py`, find the line `sys.path.insert(0, str(_PROJECT_ROOT))`. Immediately after that line, insert the pysqlite3 shim block below. Do not modify anything else in the file.
>
> ```python
> # pysqlite3 shim — required for ChromaDB on Streamlit Community Cloud.
> # Community Cloud's system sqlite3 is older than ChromaDB requires.
> # pysqlite3-binary ships a newer sqlite3; we swap it in before chromadb imports.
> # Local Mac dev skips this gracefully (pysqlite3 not installed).
> try:
>     __import__('pysqlite3')
>     import sys
>     sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
> except ImportError:
>     pass
> ```

**If you don't see the anchor:** Your `app/main.py` has been customized for Lab 2. See the **Recovery** section at the end of this notebook.


In [5]:
# VERIFY Patch 1 — grep for the pysqlite3 shim
import subprocess
from pathlib import Path

app_main = _PROJECT_ROOT / "app" / "main.py"
result = subprocess.run(
    ["grep", "-n", "pysqlite3", str(app_main)],
    capture_output=True,
    text=True,
)

assert "pysqlite3" in result.stdout, (
    "Patch 1 not applied — pysqlite3 shim missing from app/main.py"
)
print("Patch 1 anchor found in app/main.py:")
print(result.stdout)

FileNotFoundError: [WinError 2] The system cannot find the file specified

---

## Patch 2 — `app/main.py` sidebar API keys (the credit-burn fix)

### WHY

Imagine you deploy. You text the URL to your team. Three people open it.

When user A asks a question, your app calls `client.messages.create(...)`. That call needs `ANTHROPIC_API_KEY` set. If you stored YOUR key in Community Cloud Secrets, then every call user A, B, and C makes is YOUR money. Someone posts the URL on Twitter. By Tuesday morning you owe Anthropic $400.

**Public URL + your API key in environment = credit-burn.**

The fix: do not put Anthropic or Voyage keys anywhere on the server. Put a sidebar widget in the app, ask each visitor for their own keys, store them in `os.environ` only for THAT session.

Five details to notice in the patch below:
1. `type="password"` — masks the input
2. `st.stop()` — if either key is missing, Streamlit halts and renders nothing else
3. Keys go into `os.environ` only AFTER the user typed them
4. This is why Patches 3 + 4 matter — the key has to be in `os.environ` BEFORE the client is built, so the client must be built lazily (after the sidebar runs)
5. Refresh the page → keys are gone → user types them again. No persistence.

### COPY-PASTE

**Find this anchor** in `app/main.py`:

```
# DEPLOYMENT TODO (Session 4.1)
```

**Replace the entire existing comment block** (the `# ===` line above it, the `# DEPLOYMENT TODO ...` lines, and the trailing `# ============================================================` line — about 15 lines total) **with the code from the next cell:**


In [ ]:
# === COPY THIS BLOCK INTO app/main.py — DO NOT RUN THIS CELL ===
# Replaces the entire `# DEPLOYMENT TODO (Session 4.1)` comment block.

# ==============================================================
# === DEPLOYMENT: API KEYS ===
# Per-visitor key entry. No keys baked into the deployed app —
# anyone with the URL would burn the owner's credits.
# Keys live in os.environ for this session only — lost on refresh.
# ==============================================================
with st.sidebar:
    st.subheader("API Keys")
    anthropic_key = st.text_input(
        "Anthropic API Key",
        type="password",
        value=os.getenv("ANTHROPIC_API_KEY", ""),
        help="Get one at console.anthropic.com",
    )
    voyage_key = st.text_input(
        "Voyage API Key",
        type="password",
        value=os.getenv("VOYAGE_API_KEY", ""),
        help="Get one at voyageai.com",
    )

if not anthropic_key or not voyage_key:
    st.warning("Enter both API keys in the sidebar to start chatting.")
    st.stop()

os.environ["ANTHROPIC_API_KEY"] = anthropic_key
os.environ["VOYAGE_API_KEY"] = voyage_key
# === END DEPLOYMENT ===


**For Claude Code users, paste this prompt:**

> In `app/main.py`, find the comment block that starts with `# DEPLOYMENT TODO (Session 4.1)`. Replace that ENTIRE comment block (including the `# ===` separator lines above and below it) with the code below. Do not modify anything else in the file.
>
> ```python
> # ==============================================================
> # === DEPLOYMENT: API KEYS ===
> # Per-visitor key entry. No keys baked into the deployed app —
> # anyone with the URL would burn the owner's credits.
> # Keys live in os.environ for this session only — lost on refresh.
> # ==============================================================
> with st.sidebar:
>     st.subheader("API Keys")
>     anthropic_key = st.text_input(
>         "Anthropic API Key",
>         type="password",
>         value=os.getenv("ANTHROPIC_API_KEY", ""),
>         help="Get one at console.anthropic.com",
>     )
>     voyage_key = st.text_input(
>         "Voyage API Key",
>         type="password",
>         value=os.getenv("VOYAGE_API_KEY", ""),
>         help="Get one at voyageai.com",
>     )
>
> if not anthropic_key or not voyage_key:
>     st.warning("Enter both API keys in the sidebar to start chatting.")
>     st.stop()
>
> os.environ["ANTHROPIC_API_KEY"] = anthropic_key
> os.environ["VOYAGE_API_KEY"] = voyage_key
> # === END DEPLOYMENT ===
> ```

**If you don't see the anchor:** Your `app/main.py` has been customized for Lab 2. See the **Recovery** section at the end of this notebook.


In [ ]:
# VERIFY Patch 2 — grep for the new anchor
import subprocess

result = subprocess.run(
    ["grep", "-n", "=== DEPLOYMENT: API KEYS ===", str(app_main)],
    capture_output=True,
    text=True,
)

assert "DEPLOYMENT: API KEYS" in result.stdout, (
    "Patch 2 not applied — sidebar API keys block missing from app/main.py"
)
print("Patch 2 anchor found in app/main.py:")
print(result.stdout)

# Also verify the END marker
result_end = subprocess.run(
    ["grep", "-n", "=== END DEPLOYMENT ===", str(app_main)],
    capture_output=True,
    text=True,
)
assert "END DEPLOYMENT" in result_end.stdout, (
    "Patch 2 incomplete — '=== END DEPLOYMENT ===' marker missing"
)
print("Patch 2 END marker found:")
print(result_end.stdout)

# And verify the OLD TODO comment was removed
result_old = subprocess.run(
    ["grep", "-n", "DEPLOYMENT TODO (Session 4.1)", str(app_main)],
    capture_output=True,
    text=True,
)
assert not result_old.stdout.strip(), (
    "Old 'DEPLOYMENT TODO' comment is still in main.py — "
    "you only inserted the new block instead of replacing the old one. "
    "Delete the old block."
)
print("Old DEPLOYMENT TODO comment is gone (correctly replaced).")

---

## Patch 3 — `pipeline/retrieval/hyde.py` lazy client

### WHY

Open `pipeline/retrieval/hyde.py` and look at the top of the file. There are two things happening at module import time:

1. `load_dotenv(_ENV_PATH)` reads `.env` from disk
2. `client = anthropic.Anthropic()` builds an Anthropic client — and the constructor reads `ANTHROPIC_API_KEY` from `os.environ` immediately

On your laptop, this works because `.env` exists and `load_dotenv` fills `os.environ` before the client constructor runs. **On Community Cloud, there is no `.env` file.** The user types their key into the sidebar widget AFTER the app boots — but by then, `hyde.py` has already been imported, the client was built with no key, and it is a permanently broken object that will never see the key the user just typed.

**Timeline on Community Cloud:**

```
T=0  Container starts
T=1  Streamlit imports app/main.py
T=2  app/main.py imports rag.py
T=3  rag.py imports hyde.py
T=4  hyde.py runs `client = anthropic.Anthropic()`  <-- NO KEY YET. CRASH.
T=5  Sidebar widget renders                         <-- too late
T=6  User types their key                           <-- way too late
```

The fix: the **lazy client pattern** you saw in Session 3.2's `app/feedback.py`. Define a `_get_client()` function that builds the client on FIRST CALL — by then the sidebar has run, `os.environ` has the key, and the constructor sees what it needs.

**Two parts to this patch:** (a) replace the top-of-file block with a lazy `_get_client()` definition, (b) inside `generate_hypothetical_answer()`, add `client = _get_client()` immediately before the `client.messages.create(...)` call.

### COPY-PASTE — Part A: Replace the top-of-file block

**Find this anchor** in `pipeline/retrieval/hyde.py`:

```
from dotenv import load_dotenv
```

**Replace the block that starts at `from dotenv import load_dotenv` and ends at `client = anthropic.Anthropic()` (this is roughly lines 24-37) with the code from the next cell:**


In [ ]:
# === COPY THIS BLOCK INTO pipeline/retrieval/hyde.py — DO NOT RUN THIS CELL ===
# Replaces the top-of-file block from `from dotenv import load_dotenv`
# through the standalone `client = anthropic.Anthropic()` line.

import anthropic

from pipeline.embeddings.embed import embed_texts
from pipeline.ingestion.store import get_collection


def _get_client():
    """Lazy Anthropic client — instantiated on first call, after env keys are set."""
    return anthropic.Anthropic()


What you are deleting:
- `from dotenv import load_dotenv`
- The `_ENV_PATH = ...` line and its surrounding comment
- `load_dotenv(_ENV_PATH)`
- The standalone `client = anthropic.Anthropic()` line at the bottom of the block

What you are keeping (it goes back in via the new block):
- `import anthropic`
- The two `from pipeline...` imports

**For Claude Code users, paste this prompt:**

> In `pipeline/retrieval/hyde.py`, find the top-of-file block that starts at `from dotenv import load_dotenv` and ends at `client = anthropic.Anthropic()`. Replace that entire block (including the comment lines about the .env path) with the code below. Do not modify anything else in the file.
>
> ```python
> import anthropic
>
> from pipeline.embeddings.embed import embed_texts
> from pipeline.ingestion.store import get_collection
>
>
> def _get_client():
>     """Lazy Anthropic client — instantiated on first call, after env keys are set."""
>     return anthropic.Anthropic()
> ```

### COPY-PASTE — Part B: Add the lazy build inside the function

**Find this anchor** inside `generate_hypothetical_answer()` (NOT inside `hyde_retrieve()` — the API call lives in `generate_hypothetical_answer`):

```
    message = client.messages.create(
```

**Insert the line from the next cell immediately BEFORE that anchor, with the same 4-space indentation:**


In [ ]:
# === COPY THIS LINE INTO pipeline/retrieval/hyde.py — DO NOT RUN THIS CELL ===
# Goes inside `generate_hypothetical_answer()`, immediately BEFORE
# the `message = client.messages.create(...)` call. Same 4-space indent.

    client = _get_client()


After the patch, the start of `generate_hypothetical_answer()` should look like:

```python
def generate_hypothetical_answer(question: str, domain: str = "company") -> str:
    """Ask Claude to imagine what a correct answer document would say.
    ...
    """
    client = _get_client()
    message = client.messages.create(
        model="claude-sonnet-4-5",
        ...
    )
```

**For Claude Code users, paste this prompt:**

> In `pipeline/retrieval/hyde.py`, find the function `generate_hypothetical_answer`. Inside that function, find the line `message = client.messages.create(` and insert the line `client = _get_client()` immediately before it, with the same 4-space indentation. Do not insert it inside `hyde_retrieve` — only inside `generate_hypothetical_answer`.

**If you don't see the anchor:** Your `pipeline/retrieval/hyde.py` has been customized for Lab 2. See the **Recovery** section at the end of this notebook.


In [ ]:
# VERIFY Patch 3 — confirm both edits landed
import subprocess

hyde_path = _PROJECT_ROOT / "pipeline" / "retrieval" / "hyde.py"

# Check 1: _get_client function defined
result_def = subprocess.run(
    ["grep", "-n", "def _get_client", str(hyde_path)],
    capture_output=True,
    text=True,
)
assert "def _get_client" in result_def.stdout, (
    "Patch 3 Part A not applied — `def _get_client` missing from hyde.py"
)
print("Part A: _get_client() defined at:", result_def.stdout.strip())

# Check 2: client = _get_client() called inside the function
result_call = subprocess.run(
    ["grep", "-n", "client = _get_client", str(hyde_path)],
    capture_output=True,
    text=True,
)
assert "client = _get_client" in result_call.stdout, (
    "Patch 3 Part B not applied — `client = _get_client()` missing inside generate_hypothetical_answer"
)
print("Part B: client = _get_client() called at:", result_call.stdout.strip())

# Check 3: the OLD module-level client construction is gone
# (look for `client = anthropic.Anthropic()` standalone — should be 0 matches)
with open(hyde_path) as f:
    content = f.read()
assert "client = anthropic.Anthropic()" not in content, (
    "Old module-level `client = anthropic.Anthropic()` is still in hyde.py — "
    "you forgot to delete it."
)
print("Old module-level client construction is gone (correctly removed).")

# Check 4: import still works (no syntax errors introduced)
import importlib
import pipeline.retrieval.hyde
importlib.reload(pipeline.retrieval.hyde)
from pipeline.retrieval.hyde import hyde_retrieve, generate_hypothetical_answer, _get_client
print("Import still works — hyde_retrieve, generate_hypothetical_answer, _get_client all available.")

---

## Patch 4 — `pipeline/retrieval/enriched.py` lazy client

### WHY

Same problem, same fix. `pipeline/retrieval/enriched.py` also constructs an Anthropic client at module import time (line 49). Same failure mode on Community Cloud, same lazy pattern fix.

**Two parts again:** (a) replace the top-of-file block, (b) add `client = _get_client()` inside `generate_questions_for_chunk()` (NOT inside `enrich_and_store()` or `enriched_retrieve()` — the actual API call to Anthropic lives in `generate_questions_for_chunk`).

### COPY-PASTE — Part A: Replace the top-of-file block

**Find this anchor** in `pipeline/retrieval/enriched.py`:

```
from dotenv import load_dotenv
```

**Replace the block that starts at `from dotenv import load_dotenv` and ends at `client = anthropic.Anthropic()` (roughly lines 36-49) with the code from the next cell:**


In [ ]:
# === COPY THIS BLOCK INTO pipeline/retrieval/enriched.py — DO NOT RUN THIS CELL ===
# Replaces the top-of-file block from `from dotenv import load_dotenv`
# through the standalone `client = anthropic.Anthropic()` line.
# NOTE: keep the `import chromadb` line — it's part of the new block below.

import anthropic
import chromadb

from pipeline.embeddings.embed import embed_texts
from pipeline.ingestion.store import get_collection, CHROMA_PATH


def _get_client():
    """Lazy Anthropic client — instantiated on first call, after env keys are set."""
    return anthropic.Anthropic()


Note that `enriched.py` also imports `chromadb` at this same block — keep that import.

What you are deleting:
- `from dotenv import load_dotenv`
- The `_ENV_PATH = ...` line and its surrounding comment
- `load_dotenv(_ENV_PATH)`
- The standalone `client = anthropic.Anthropic()` line

**For Claude Code users, paste this prompt:**

> In `pipeline/retrieval/enriched.py`, find the top-of-file block that starts at `from dotenv import load_dotenv` and ends at `client = anthropic.Anthropic()`. Replace that entire block (including the comment lines about the .env path) with the code below. Keep the `import chromadb` line — it's part of the new block. Do not modify anything else in the file.
>
> ```python
> import anthropic
> import chromadb
>
> from pipeline.embeddings.embed import embed_texts
> from pipeline.ingestion.store import get_collection, CHROMA_PATH
>
>
> def _get_client():
>     """Lazy Anthropic client — instantiated on first call, after env keys are set."""
>     return anthropic.Anthropic()
> ```

### COPY-PASTE — Part B: Add the lazy build inside the function

**Find this anchor** inside `generate_questions_for_chunk()`:

```
    message = client.messages.create(
```

**Insert the line from the next cell immediately BEFORE that anchor, with the same 4-space indentation:**


In [ ]:
# === COPY THIS LINE INTO pipeline/retrieval/enriched.py — DO NOT RUN THIS CELL ===
# Goes inside `generate_questions_for_chunk()`, immediately BEFORE
# the `message = client.messages.create(...)` call. Same 4-space indent.

    client = _get_client()


After the patch, the body of `generate_questions_for_chunk()` near the API call should look like:

```python
    # ------------------------------------------------------------------
    # Lab 2 breadcrumb: ...
    # ------------------------------------------------------------------
    client = _get_client()
    message = client.messages.create(
        model="claude-sonnet-4-5",
        ...
    )
```

**For Claude Code users, paste this prompt:**

> In `pipeline/retrieval/enriched.py`, find the function `generate_questions_for_chunk`. Inside that function, find the line `message = client.messages.create(` and insert the line `client = _get_client()` immediately before it, with the same 4-space indentation. Do not insert it inside `enrich_and_store` or `enriched_retrieve` — only inside `generate_questions_for_chunk`.

**If you don't see the anchor:** Your `pipeline/retrieval/enriched.py` has been customized for Lab 2. See the **Recovery** section at the end of this notebook.


In [ ]:
# VERIFY Patch 4 — confirm both edits landed
import subprocess

enriched_path = _PROJECT_ROOT / "pipeline" / "retrieval" / "enriched.py"

# Check 1: _get_client function defined
result_def = subprocess.run(
    ["grep", "-n", "def _get_client", str(enriched_path)],
    capture_output=True,
    text=True,
)
assert "def _get_client" in result_def.stdout, (
    "Patch 4 Part A not applied — `def _get_client` missing from enriched.py"
)
print("Part A: _get_client() defined at:", result_def.stdout.strip())

# Check 2: client = _get_client() called inside the function
result_call = subprocess.run(
    ["grep", "-n", "client = _get_client", str(enriched_path)],
    capture_output=True,
    text=True,
)
assert "client = _get_client" in result_call.stdout, (
    "Patch 4 Part B not applied — `client = _get_client()` missing inside generate_questions_for_chunk"
)
print("Part B: client = _get_client() called at:", result_call.stdout.strip())

# Check 3: the OLD module-level client construction is gone
with open(enriched_path) as f:
    content = f.read()
assert "client = anthropic.Anthropic()" not in content, (
    "Old module-level `client = anthropic.Anthropic()` is still in enriched.py — "
    "you forgot to delete it."
)
print("Old module-level client construction is gone (correctly removed).")

# Check 4: import still works
import importlib
import pipeline.retrieval.enriched
importlib.reload(pipeline.retrieval.enriched)
from pipeline.retrieval.enriched import enriched_retrieve, generate_questions_for_chunk, _get_client
print("Import still works — enriched_retrieve, generate_questions_for_chunk, _get_client all available.")

---

## Patch 5 — `app/feedback.py` (NO CHANGE — confirmation only)

Open `app/feedback.py`. You wrote (or were handed) this file in Session 3.2.

Look at:
- `_get_client()` (line ~28) — already lazy. Returns `None` if Phoenix is unavailable.
- `submit_feedback()` (line ~38) — calls `client.spans.add_span_annotation(...)` wrapped in try/except. If Phoenix is down, the call silently no-ops.
- `get_feedback_summary()` (line ~68) — also wrapped in try/except.

**No code change needed.** This is exactly the property we want for production: if the observability backend is unreachable, the chat keeps working. Tracing silently falls off, but the user-facing app does not crash.

Run the cell below to inspect the current `feedback.py` and confirm both Phoenix calls are wrapped.

In [ ]:
# VERIFY Patch 5 — confirm feedback.py already has the right pattern
import subprocess

feedback_path = _PROJECT_ROOT / "app" / "feedback.py"

# Confirm both Phoenix calls exist
calls = subprocess.run(
    ["grep", "-n", "-E", r"add_span_annotation|get_spans_dataframe", str(feedback_path)],
    capture_output=True,
    text=True,
)
print("Phoenix calls found in app/feedback.py:")
print(calls.stdout)

# Confirm try/except wrappers exist
tries = subprocess.run(
    ["grep", "-n", "-E", r"try:|except Exception", str(feedback_path)],
    capture_output=True,
    text=True,
)
print("\ntry/except blocks found in app/feedback.py:")
print(tries.stdout)

print("Patch 5 confirmed — Phoenix calls are wrapped. No change needed.")

---

## Final Patch Check — `git diff`

Before we move to smoke testing, run `git diff` in your terminal. You should see edits ONLY in:

- `app/main.py` (Patches 1 + 2)
- `pipeline/retrieval/hyde.py` (Patch 3)
- `pipeline/retrieval/enriched.py` (Patch 4)

If any other file shows up in `git diff`, you went off-script. Reset that file with `git checkout -- <file>` and re-apply only the listed patches.

---

## Section 3 — Smoke Test (`streamlit run` + `validate_deploy.py`)

You've applied 5 patches. Before deploying anywhere, prove they actually work.

Two checks, both fast:

1. **Patch validation** — `scripts/validate_deploy.py` greps for every patch anchor and confirms `hyde.py`, `enriched.py`, and `app.rag` import cleanly **without** `ANTHROPIC_API_KEY` in the environment. That last bit is the real test of the lazy-client patches: if Patch 3 or 4 missed a step, the import fails here.

2. **Streamlit smoke test** — move your `.env` aside and run `streamlit run app/main.py`. The sidebar should appear with two empty password fields. The chat should NOT appear until you enter both keys. Once you enter them, ask one question and confirm a Phoenix trace lands.

If both pass, you're cleared to deploy. If either fails, fix it before pushing — Community Cloud is harder to debug.


In [8]:
# Run the validation script (free — no API calls).
# Exits 0 if every patch landed and the modules import without keys.
import subprocess
result = subprocess.run(
    ["uv", "run", "python", "C:/AI-2/ai3-fullstack/scripts/validate_deploy.py"],
    cwd="..",
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("--- STDERR ---")
    print(result.stderr)
    raise RuntimeError("Patch validation failed — see output above. Fix before deploying.")


Validating Session 4.1 deploy patches...

Patch 1 - app/main.py pysqlite3 shim:
  PASS  pysqlite3 import shim

Patch 2 - app/main.py sidebar API keys:
  PASS  sidebar block anchor
  PASS  ANTHROPIC env write
  PASS  VOYAGE env write

Patch 3 - pipeline/retrieval/hyde.py lazy client:
  PASS  lazy _get_client function
  PASS  in-function client = _get_client() call
  PASS  no module-level client construction

Patch 4 - pipeline/retrieval/enriched.py lazy client:
  PASS  lazy _get_client function
  PASS  in-function client = _get_client() call
  PASS  no module-level client construction

Patch 5 - app/feedback.py (no change required, just confirming the existing pattern):
  PASS  try/except wrapping present

Import check (the real test â€” patched modules load WITHOUT API keys):
  PASS  pipeline.retrieval.hyde imports without API key
  PASS  pipeline.retrieval.enriched imports without API key
  PASS  app.rag imports without API key

All 5 patches applied. You're ready to deploy to Communi

### Streamlit smoke test (manual — instructor-led)

In a terminal at the repo root:

```bash
mv .env .env.backup     # hide your local keys to simulate the deployed app
uv run streamlit run app/main.py
```

**Pass criteria — all four must be true:**

- [ ] Sidebar appears with two empty password fields (`Anthropic API Key`, `Voyage API Key`).
- [ ] Chat input is blocked until both keys are entered (warning message visible).
- [ ] After entering keys, a question returns sources and an answer.
- [ ] Phoenix shows a trace under your `PHOENIX_PROJECT_NAME` within 5-10 seconds.

When the smoke test passes:

```bash
# stop streamlit (Ctrl+C), then restore your env so other tools work
mv .env.backup .env
```

> If streamlit fails: most common cause is Patch 2 placed AFTER `from app.rag import get_response`. The sidebar must run BEFORE any code that needs the keys.


## Section 5 — Deploy to Streamlit Community Cloud

Five steps. Walk through them with the instructor.

### STEP 1 — Push your branch to your fork

```bash
git add app/main.py pipeline/retrieval/hyde.py pipeline/retrieval/enriched.py
git commit -m "Apply Session 4.1 deployment patches"
git push fork <your-branch>
```

Your branch must already exist on your fork (this was in the pre-class email). If `fork` is not a remote, set it up:
```bash
git remote add fork https://github.com/<your-handle>/ai3-fullstack.git
git push -u fork <your-branch>
```

### STEP 2 — Create the app on share.streamlit.io

1. Open [share.streamlit.io](https://share.streamlit.io).
2. Sign in with GitHub. Grant access to your fork if prompted.
3. Click the blue **Create app** button.
4. Fill in:
   - **Repository:** `<your-handle>/ai3-fullstack`
   - **Branch:** `<your-branch>` (the one you just pushed)
   - **Main file path:** `app/main.py`
   - **App URL:** `<your-handle>-northbrook` (or anything unique to you)

### STEP 3 — Paste Phoenix Secrets

Click **Advanced settings**. There is a Secrets textbox.

Paste this (filled in with YOUR project name and the Phoenix key the instructor shared during break):

```toml
PHOENIX_API_KEY = "<the-key-instructor-pasted-in-chat>"
PHOENIX_COLLECTOR_ENDPOINT = "https://app.phoenix.arize.com/s/tyler-hayes"
PHOENIX_PROJECT_NAME = "ai3-<your-handle>"
```

**Phoenix triple ONLY.** Do NOT paste Anthropic or Voyage keys here. Those go in the sidebar (Patch 2).

Reference the file `.streamlit/secrets.toml.example` in your repo — it has this template.

### STEP 4 — Deploy

Click **Deploy**. The page shows logs. First deploy takes 1-3 minutes — it installs dependencies and builds the corpus into ChromaDB.

When the page flips to your app, you should see the sidebar with two empty key fields and the warning: *"Enter both API keys in the sidebar to start chatting."* That means it worked.

Now:
1. Enter your Anthropic key in the sidebar.
2. Enter your Voyage key in the sidebar.
3. The chat should appear.
4. Ask a question. Wait for the answer.

### STEP 5 — Verify Phoenix

Open [app.phoenix.arize.com/s/tyler-hayes](https://app.phoenix.arize.com/s/tyler-hayes) in a new tab.

Find your project — `ai3-<your-handle>` — in the project dropdown. You should see a trace for the question you just asked. Spans for retrieval, generation, the whole pipeline.

If you see the trace, you are done. **Public URL. Real deployment. Phoenix observability.**

Paste your URL in class chat so the instructor can see five lights green.

### Troubleshooting (top 5 deploy failures)

| Error | Fix |
|-------|-----|
| Repo not visible in `share.streamlit.io` dashboard | Reauthorize GitHub OAuth in Streamlit settings. Free tier needs the repo to be public OR you grant explicit access. |
| Build error mentioning `pysqlite3-binary` | Open `requirements.txt`. Confirm the line is `pysqlite3-binary>=0.5.0; sys_platform == "linux"`. The platform marker is what skips it on Mac dev. |
| App boots but chat shows auth error | Sidebar key entry did not run. Patch 2 is in the wrong place. Look at `app/main.py` and confirm the `# === DEPLOYMENT: API KEYS ===` block sits BEFORE Step 5 (the chat input handler). |
| Traces do not appear in Phoenix | Wait 5-10 seconds — `BatchSpanProcessor` flushes async. Refresh Phoenix. Check the project dropdown — does the name match `PHOENIX_PROJECT_NAME` in your Secrets? Capitalization matters. |
| App sleeps and wakes slowly | 12-hour inactivity timeout. Pre-warm the URL right before the 4.2 demo by visiting it. |

---

## Optional Appendix — Run with Docker locally

You don't need this for class. Community Cloud handles the container build for you. But if you want to test the app in a real container on your laptop — say, before showing it to a friend, or to debug a Community Cloud issue locally — the launcher scripts are in `scripts/`.

| Platform | Command |
|----------|---------|
| Mac / Linux | `./scripts/run.sh` |
| Windows (any PowerShell version) | `.\scripts\run.cmd` |

The first build takes 5-10 minutes (Docker pulls the Python base image and installs everything in `requirements.txt`). After that, runs are cached and start in seconds.

The launcher scripts handle:
- Stale-container cleanup from prior crashed runs
- Port-in-use detection (override with `PORT=8502 ./scripts/run.sh`)
- Forward-slash mount path conversion on Windows
- Trap-stop on Ctrl+C

If Docker fails to start, ensure Docker Desktop is running. The container does **not** need a `.env` file — it uses the same sidebar key entry pattern as the deployed app.


## Section 6 — Recovery (if your file diverged from the canonical version)

If `git diff` shows you customized `app/main.py`, `pipeline/retrieval/hyde.py`, or `pipeline/retrieval/enriched.py` for Lab 2 and the anchors no longer match, use the canonical regions below to reconcile by hand.

**Workflow:**
1. Find the rough region in your file.
2. Compare against the canonical version below.
3. Apply the deployment patch to YOUR version, preserving any Lab 2 customizations.
4. Run the verify cells again to confirm.

### Canonical `app/main.py` deployment region (after all patches)

```python
_PROJECT_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(_PROJECT_ROOT))

# pysqlite3 shim — required for ChromaDB on Streamlit Community Cloud.
# Community Cloud's system sqlite3 is older than ChromaDB requires.
# pysqlite3-binary ships a newer sqlite3; we swap it in before chromadb imports.
# Local Mac dev skips this gracefully (pysqlite3 not installed).
try:
    __import__('pysqlite3')
    import sys
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
except ImportError:
    pass

import os
import uuid

import streamlit as st
import yaml
from app.branding import apply_branding
from app.feedback import submit_feedback, get_feedback_summary
from app.rag import get_response

from dotenv import load_dotenv
load_dotenv(_PROJECT_ROOT / ".env")

# ... Phoenix init, session_id init (unchanged from 3.2) ...

# ==============================================================
# === DEPLOYMENT: API KEYS ===
# Per-visitor key entry. No keys baked into the deployed app —
# anyone with the URL would burn the owner's credits.
# Keys live in os.environ for this session only — lost on refresh.
# ==============================================================
with st.sidebar:
    st.subheader("API Keys")
    anthropic_key = st.text_input(
        "Anthropic API Key",
        type="password",
        value=os.getenv("ANTHROPIC_API_KEY", ""),
        help="Get one at console.anthropic.com",
    )
    voyage_key = st.text_input(
        "Voyage API Key",
        type="password",
        value=os.getenv("VOYAGE_API_KEY", ""),
        help="Get one at voyageai.com",
    )

if not anthropic_key or not voyage_key:
    st.warning("Enter both API keys in the sidebar to start chatting.")
    st.stop()

os.environ["ANTHROPIC_API_KEY"] = anthropic_key
os.environ["VOYAGE_API_KEY"] = voyage_key
# === END DEPLOYMENT ===
```

### Canonical `pipeline/retrieval/hyde.py` (after Patch 3)

```python
"""
hyde.py -- Hypothetical Document Embeddings (HyDE)
...
"""

from pathlib import Path

import anthropic

from pipeline.embeddings.embed import embed_texts
from pipeline.ingestion.store import get_collection


def _get_client():
    """Lazy Anthropic client — instantiated on first call, after env keys are set."""
    return anthropic.Anthropic()


def generate_hypothetical_answer(question: str, domain: str = "company") -> str:
    """..."""
    client = _get_client()
    message = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=256,
        system=(...),
        messages=[...],
    )
    return message.content[0].text


def hyde_retrieve(question: str, top_k: int = 5, domain: str = "company") -> list[dict]:
    # ... unchanged ...
```

### Canonical `pipeline/retrieval/enriched.py` (after Patch 4)

```python
"""
enriched.py -- Question Enrichment at Ingestion Time
...
"""

from pathlib import Path

import anthropic
import chromadb

from pipeline.embeddings.embed import embed_texts
from pipeline.ingestion.store import get_collection, CHROMA_PATH


def _get_client():
    """Lazy Anthropic client — instantiated on first call, after env keys are set."""
    return anthropic.Anthropic()


ENRICHED_COLLECTION = "northbrook_enriched"
N_QUESTIONS_PER_CHUNK = 3


def generate_questions_for_chunk(chunk_text: str, n_questions: int = N_QUESTIONS_PER_CHUNK) -> list[str]:
    """..."""
    client = _get_client()
    message = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=256,
        temperature=0,
        messages=[...],
    )
    # ... unchanged ...


def enrich_and_store(...):
    # ... unchanged ...


def enriched_retrieve(...):
    # ... unchanged ...
```

---

## Wrap — What You Built Today

- **Patches 1-5** applied to your working branch — pysqlite3 shim, sidebar API keys, two lazy clients, one no-op confirmation
- **Local streamlit smoke test** — sidebar gates, query returns sources, Phoenix trace lands
- **Local Docker smoke test** — same flow inside a container
- **Live Community Cloud deploy** — public URL, Phoenix Secrets configured, traces under your project name

### The three problems you solved

1. **"Works on my machine"** → Docker (already done — you used the existing `Dockerfile`)
2. **Module-level state breaks containers** → Lazy client pattern (Patches 3 + 4)
3. **Public deploy burns credits** → Sidebar API key entry (Patch 2)

### Final Project — assigned now, due Session 4.2

The deployed URL you stood up today is the foundation of the **Final Project** (15% of your grade). Same artifact as Lab 2; two grades on one deployable.

**At Session 4.2 you will demo your live URL** — 8 minutes per student. Plan to show:
- One good Q&A with sources
- One injection attempt blocked by your guards
- One feedback capture (thumbs up/down)
- A Phoenix trace appearing live in the workspace
- One question the instructor poses you did not prepare for

Polish your branding. Practice with a timer. Pre-warm your URL the morning of 4.2 so it does not boot from cold while you are presenting.

### Questions to leave with

> **On Reflection:** "What is the most important thing you learned in this course?"

> **On Growth:** "What would you build differently if you started over?"

We will discuss these at the start of 4.2 before demos begin.

---

### Done state

Before you leave today, confirm you have ALL of the following:

- [ ] Patches 1-5 applied (verified by the cells above)
- [ ] Local Docker smoke test passing
- [ ] Live Community Cloud URL
- [ ] Phoenix traces appearing under your project name

If any of those are not true, finish before next session — that is the Final Project.